##**Quantitative Translational Imaging in Medicine Lab — Summer Scholar Program**

<div style="background-color:black; padding:20px; text-align:center; border-radius: 8px;">
    <img src="https://i0.wp.com/www.martinos.org/wp-content/uploads/2019/01/spark_no_fade.gif?fit=404%2C303&ssl=1" alt="Martinos Center Logo" width="400"/>
    <h1 style="color:white; font-family: sans-serif;">Martinos Center for Biomedical Imaging</h1>
</div>

# Day 1: What is a medical image?

**Today's goal:** by the end of this notebook, you'll understand that every medical image
— MRI, CT, ultrasound, even a photo on your phone — is just a **grid of numbers**. Once you
see that, you can start to understand what radiologists and researchers actually
look at, and how software can analyze images automatically.

HOW-TO:
Run each cell in order (click the cell, then click the play button or press **Shift+Enter**).

## 0. A 5-minute Python refresher

Before we touch any images, here are the few building blocks of Python we'll use all week.
If any of this is unfamiliar, that's completely fine — just run the cells and watch what
happens.

**A variable** is just a name that stores a value, like a labeled box.

In [ ]:
# This creates a variable called my_number and stores the value 7 in it
my_number = 7

# print() displays a value so we can see it
print(my_number)

**A list** is an ordered collection of values, written with square brackets `[ ]`.
You can access one item in a list using its position (starting from 0, not 1!).

In [ ]:
my_list = [10, 20, 30, 40]

print(my_list[0])   # the FIRST item (position 0)
print(my_list[2])   # the THIRD item (position 2)

**A function** is a reusable piece of code that takes some input(s) and does something
with them. You "call" a function by writing its name followed by parentheses containing its
inputs.

In [ ]:
# round() is a built-in function - it takes a number and rounds it
print(round(3.14159, 2))   # round to 2 decimal places

# len() is a built-in function - it tells you how many items are in a list
print(len(my_list))

**An import** brings in extra tools that aren't built into Python by default. We'll use
this constantly — almost every notebook this week starts with a cell of imports.

In [ ]:
# numpy (nicknamed "np") is a library for working with grids of numbers - we'll use it heavily
import numpy as np

# matplotlib (nicknamed "plt") is a library for making plots and displaying images
import matplotlib.pyplot as plt

print("Libraries loaded. Ready to go!")

That's genuinely all the Python vocabulary you need to get started. Everything else
we'll explain inline as it comes up.

## 1. An image is just a grid of numbers

A black-and-white photo is a 2D grid (a "matrix"). Each little square is a **pixel**, and
each pixel has a number that says how bright it is: 0 means black, 1 means white, and numbers
in between are shades of gray.

Let's build a tiny image by hand — a 5x5 grid of numbers we pick ourselves.

In [ ]:
# np.array() turns a list of lists into a grid numpy can work with
# Each inner [ ] is one ROW of the image
tiny_image = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 1.0, 1.0, 0.0],
    [0.0, 1.0, 0.5, 1.0, 0.0],
    [0.0, 1.0, 1.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0],
])

print(tiny_image)

Now let's actually turn those numbers into a picture using `plt.imshow()`
("image show").

In [ ]:
plt.imshow(tiny_image, cmap='gray', vmin=0, vmax=1)
plt.colorbar(label='pixel value')
plt.title('A 5x5 image')
plt.show()

**Predict before you run:** if you changed the middle number (currently `0.5`) to `0.0`,
what do you think would happen to the picture? Write down your guess, then try it in the
next cell.

In [ ]:
# Try your own pattern here! Change some numbers (must be between 0.0 and 1.0) and re-run.
my_image = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0],
])

plt.imshow(my_image, cmap='gray', vmin=0, vmax=1)
plt.colorbar(label='pixel value')
plt.show()

**Reflection:** in a sentence or two, explain to someone who has never coded what a
digital image actually is.

In [ ]:
# Write your explanation here as a comment (a line starting with #) - it won't run as code,
# it's just a note for yourself.

#
#

## 2. Real images are just bigger grids

A real medical image isn't 5x5 — it might be 256x256, or 512x512. That's a LOT of numbers
(256x256 = 65,536 of them!), but the idea is exactly the same. Let's look at a
*simulated* brain MRI slice.

> **Note:** the images this week are realistic *simulations*, not real patient scans — that
> keeps things simple for now. You'll get to see medical imaging data from public datasets later in the week.

In [ ]:
# If you're running this in Google Colab, run this cell first to make sure
# scikit-image is installed. If you're running locally and already have it, this is
# harmless - it'll just say "already satisfied" and do nothing.
!pip install -q scikit-image

In [ ]:
from scipy.ndimage import gaussian_filter
from skimage.data import shepp_logan_phantom
from skimage.transform import resize

def make_synthetic_brain_mri(size=256, seed=1, add_tumor=True):
    rng = np.random.default_rng(seed)
    y, x = np.ogrid[:size, :size]
    cy, cx = size/2, size/2
    r = np.sqrt((y-cy)**2 + (x-cx)**2)
    img = np.zeros((size, size))
    R = size*0.42
    img[(r < R) & (r > R*0.92)] = 0.95
    tissue_mask = r < R*0.90
    base = 0.55 + 0.05*gaussian_filter(rng.standard_normal((size,size)), 8)
    img[tissue_mask] = base[tissue_mask]
    white_mask = r < R*0.60
    img[white_mask] = 0.75 + 0.03*gaussian_filter(rng.standard_normal((size,size)), 8)[white_mask]
    for sign in [-1, 1]:
        ey, ex = cy, cx + sign*size*0.09
        ell = ((y-ey)/(size*0.10))**2 + ((x-ex)/(size*0.05))**2
        img[ell < 1] = 0.15
    if add_tumor:
        ty, tx = cy - size*0.15, cx + size*0.20
        ell = ((y-ty)/(size*0.045))**2 + ((x-tx)/(size*0.055))**2
        img[ell < 1] = 0.92
    img = gaussian_filter(img, 1.0)
    img += rng.normal(0, 0.02, img.shape)
    img[r >= R] = np.clip(img[r >= R], 0, 0.03)
    return np.clip(img, 0, 1)

def make_ct_phantom(size=256):
    p = shepp_logan_phantom()
    p = resize(p, (size, size), anti_aliasing=True)
    hu = -1000 + p * 5000
    return np.clip(hu, -1000, 3000)

def make_ultrasound(size=256, seed=2):
    rng = np.random.default_rng(seed)
    y, x = np.ogrid[:size, :size]
    depth_gain = np.clip(1.2 - (y/size)*0.8, 0.2, 1.2)
    base = 0.4 * depth_gain * np.ones((size, size))
    speckle = rng.gamma(shape=4, scale=0.25, size=(size, size))
    img = base * speckle
    cy, cx, cr = size*0.55, size*0.5, size*0.08
    r = np.sqrt((y-cy)**2 + (x-cx)**2)
    img[r < cr] = rng.gamma(4, 0.03, img.shape)[r < cr]
    enhance_mask = (x > cx-cr) & (x < cx+cr) & (y > cy+cr) & (y < cy+cr*3)
    img[enhance_mask] *= 1.4
    img = gaussian_filter(img, 0.7)
    return np.clip(img, 0, 1.5)

print("Image-generating functions ready!")

# Calling the function generates one simulated MRI image
mri = make_synthetic_brain_mri()

# .shape tells us the size of the grid: (rows, columns)
print("Shape of this image:", mri.shape)

In [ ]:
plt.figure(figsize=(5,5))
plt.imshow(mri, cmap='gray')
plt.title('Simulated MRI slice')
plt.colorbar(label='signal intensity')
plt.show()

## 3. Zooming in on the actual numbers

Let's prove to ourselves this is really just numbers by printing a small patch of the image.
`mri[a:b, c:d]` means "grab the rows from a to b, and the columns from c to d" — this is
called **slicing**.

In [ ]:
# Grab a small 6x6 patch from near the center of the image
center = mri.shape[0] // 2   # // means divide and round down

patch = mri[center-3:center+3, center-3:center+3]

# np.round rounds every number in the grid to 2 decimal places, just to make it easier to read
print(np.round(patch, 2))

In [ ]:
# Now let's see that same patch as a picture, with the numbers overlaid
plt.imshow(patch, cmap='gray', vmin=0, vmax=1)
plt.title('Zoomed in — 6x6 pixels')

# this loop writes each number onto its own pixel in red text
for i in range(6):
    for j in range(6):
        plt.text(j, i, f"{patch[i,j]:.2f}", ha='center', va='center', color='red', fontsize=8)
plt.show()

## 4. Comparing three imaging modalities

Different imaging machines measure completely different physical things, but they all produce
a grid of numbers in the end:

- **MRI** measures how hydrogen atoms (mostly in water and fat) respond to magnetic fields and
  radio waves.
- **CT** measures how much X-rays are absorbed as they pass through the body (denser things
  like bone absorb more).
- **Ultrasound** measures echoes of sound waves bouncing off tissue boundaries.

In [ ]:
ct = make_ct_phantom()
us = make_ultrasound()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(mri, cmap='gray')
axes[0].set_title('MRI\n(measures H atoms in a magnetic field)')

axes[1].imshow(ct, cmap='gray')
axes[1].set_title('CT\n(measures X-ray absorption)')

axes[2].imshow(us, cmap='gray')
axes[2].set_title('Ultrasound\n(measures sound echoes)')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

**Reflection**
1. Which image looks "cleanest" / least noisy? Why do you think that is?
2. The MRI and CT images have a bright white ring around the outside — any guesses what that
   represents?
3. The ultrasound image has a grainy, speckled texture. Real ultrasound images always look
   like this — it's called **speckle**, a physical side-effect of sound waves interfering with
   each other, not a flaw in the picture.

## 5. CT and "Hounsfield Units" — numbers with real meaning

Unlike a regular photo, CT pixel values aren't arbitrary — they're a real physical
measurement called **Hounsfield Units (HU)**, calibrated so that:

| Tissue | Typical HU |
|---|---|
| Air | -1000 |
| Fat | -100 to -50 |
| Water | 0 |
| Soft tissue (muscle, organs) | +10 to +60 |
| Bone | +400 to +1000 |

This means a radiologist (or a computer program) can look at the actual pixel *value* and
know what kind of tissue it probably is.

In [ ]:
# Let's check the HU value at a few specific spots.
# ct[row, col] grabs the single number at that exact position.

print("Background (air), top-left corner:", ct[10, 10], "HU")
print("Center of the image:", ct[ct.shape[0]//2, ct.shape[1]//2], "HU")

**Try it:** pick your own row and column (both between 0 and 255) in the cell below.
Can you find a spot that lands in the "bone" range from the table above?

In [ ]:
# Your turn - try some coordinates!
row = 100   # <- change this number (0-255)
col = 100   # <- change this number (0-255)

print(f"HU value at row={row}, col={col}:", ct[row, col])

## 6. Windowing — how radiologists actually "adjust" images

Here's something powerful: the SAME image can look completely different depending on what
range of values you choose to display. This is called **windowing**. Let's build it up one
step at a time.

In [ ]:
# vmin and vmax control what range of numbers gets mapped to black vs. white.
# Wide window: shows the full range of values at once.
plt.imshow(ct, cmap='gray', vmin=-1000, vmax=1500)
plt.title('Wide window (-1000 to 1500)')
plt.colorbar()
plt.show()

In [ ]:
# Soft tissue window: zoom into the soft-tissue range. Bone and air both become
# solid colors (saturated) because they're outside this narrower range.
plt.imshow(ct, cmap='gray', vmin=-160, vmax=240)
plt.title('Soft tissue window (-160 to 240)')
plt.colorbar()
plt.show()

In [ ]:
# Bone window: zoom into the bone range instead.
plt.imshow(ct, cmap='gray', vmin=300, vmax=2000)
plt.title('Bone window (300 to 2000)')
plt.colorbar()
plt.show()

**Note:** no pixel value actually changed between these three pictures —
only the *range we chose to display* changed. Radiologists adjust the "window" constantly
while reading scans to bring out the tissue they care about.

**Try it:** go back to any of the three cells above and change the `vmin`/`vmax` numbers.
Can you make a window where the bright ring disappears into solid white? Can you make one
where you can barely see anything at all?

## Wrap-up: What you learned today

- Every digital image, including medical images, is a grid of numbers (pixels)
- MRI, CT, and ultrasound all measure different physical phenomena, but all produce
  number-grids in the end
- CT values (Hounsfield Units) are physically meaningful and standardized
- "Windowing" lets you change how an image *looks* without changing the underlying data

**Journal prompt** (2-4 sentences): What's one thing from today that surprised you, or
that you'd want to explain to a friend who has never thought about how medical images work?

**Tomorrow:** we'll dig into more detail in how MRI, CT and ultrasound work

In [ ]:
# Journal entry:
#